# Fluid model QC (standard template)

Run by `opmagent` for every fluid model version; the HTML export is the version's QC report and the approval is tied to its sha256. **Read only:** the notebook loads the build payload and the tables from the database, computes checks and plots, and writes `qc_summary.json`. Don't edit this template for one model: write an investigation notebook instead (`opmagent fluid investigate`).

In [ ]:
payload_path = "payload.json"   # written by the fluid pipeline
db_path = ""                    # SQLite database (opened read-only)
version_id = 0                  # fluid_model_versions.id
summary_path = "qc_summary.json"

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'   # self-contained HTML export
import pandas as pd
from IPython.display import display
from opmagent.notebooks import qc_lib as qc

data = qc.load(payload_path, db_path, version_id)
findings = qc.run_all_checks(data)
qc.header(data)

## Summary

Tuning checks against the configured tolerances (these gate the build), then every QC finding, worst first.

In [ ]:
display(qc.show_tuning(data))
qc.show_findings(findings)

## Assumptions and validator warnings

In [ ]:
display(qc.show_list(data.get('assumptions', []), title='Assumptions'))
qc.show_list(title='Validator warnings', items=[w['message'] + (f" ({w['where']})" if w.get('where') else '') for w in data.get('validator_warnings', [])])

## 1. Lab data

Composition sum, DL monotonicity and the CCE break against the lab saturation pressure. The characterised composition should continue the C6- trend into the pseudo-components.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] == 'lab data']))
from opmagent.pvt import engine
spec = qc.fluid_spec(data)
qc.fig_composition(engine.components(spec))

## 2. Phase envelope

Reservoir temperature and saturation pressure (model and lab) and the separator path.

In [ ]:
try:
    env = engine.phase_envelope(spec)
    display(qc.fig_phase_envelope(env, data))
except Exception as e:  # NeqSim may fail to trace some envelopes; that is a finding, not a crash
    findings.append(qc.finding('lab vs model', 'Phase envelope', 'warn', f'could not trace: {e}'))
    print('Phase envelope could not be traced:', e)

## 3. Lab experiments: measured vs model

Curves are simulated on a dense pressure grid. The error tables compare at the lab's pressures; for DL the model is re-run on the lab's own steps, because differential liberation depends on the step sequence.

In [ ]:
lab, model = data['lab'], data['model']
if model.get('cce'):
    display(qc.fig_lab_vs_model('CCE relative volume', 'Pressure (bar)', 'V / Vsat', lab.get('cce'), 'relative_volume', model['cce'], 'relative_volume', 'V/Vsat'))
    df = qc.match_frame(lab.get('cce') or [], model['cce'], 'relative_volume', 'relative_volume')
    if len(df):
        display(df.round(4))
if model.get('dl'):
    display(qc.fig_dl(data))
    for lk, mk in (('bo', 'bo_d'), ('rs', 'rs_d'), ('oil_density', 'oil_density')):
        df = qc.match_frame(lab.get('dl') or [], qc.dl_model_rows(data), lk, mk)
        if len(df):
            print(f'DL {lk}'); display(df.round(4))
if model.get('viscosity'):
    display(qc.fig_lab_vs_model('Oil viscosity (reservoir fluid)', 'Pressure (bar)', 'Viscosity (cP)', lab.get('viscosity'), 'mu_o', model['viscosity'], 'mu_o', 'mu_o'))
    df = qc.match_frame(lab.get('viscosity') or [], model['viscosity'], 'mu_o', 'mu_o')
    if len(df):
        display(df.round(4))
qc.show_findings([f for f in findings if f['area'] == 'lab vs model'])

## 4. Black-oil tables as stored in the database

These are the rows the deck builder writes into PVTO / PVDG / PVTW / DENSITY, checked for what OPM Flow needs.

In [ ]:
t = data['tables']
if t.get('pvto'):
    display(qc.fig_pvto(t['pvto']))
if t.get('pvdg'):
    display(qc.fig_pvdg(t['pvdg']))
for name in ('pvtw', 'density', 'rsvd'):
    if t.get(name):
        print(name.upper()); display(pd.DataFrame(t[name]).drop(columns='version_id'))
qc.show_findings([f for f in findings if f['area'] == 'tables'])

## Result

In [ ]:
summary = qc.write_summary(summary_path, findings, data)
print(f"Overall: {summary['overall'].upper()}  " + ", ".join(f'{k} {v}' for k, v in summary['counts'].items()))